# 03 团投影（high-order → pairwise）

`honet.clique_projection(hg)` 把超图展开成成对网络：每条大小为 `k` 的超边在其成员之间生成一个 `k` 团，
返回 `dict[结点, frozenset[邻居]]`。

它与 `Hypergraph.neighbors` 的定义一致（`N(i) = (∪_{e ∋ i} e) \ {i}`），区别是前者一次性给出全部结点的映射。

**预期看到**：投影结果与逐点查询邻居相同；同时能观察到**信息损失**——三条二元边 `{a,b},{b,c},{a,c}`
与一条三元边 `{a,b,c}` 的投影完全一样，但原图的高阶结构不同。

这正是“高阶网络 vs 成对网络”的核心差别：投影后的图无法区分“三人共同出现”与“两两各自合作”。

In [1]:
import honet

hg = honet.Hypergraph(edges=[["a", "b", "c"], ["c", "d", "e"], ["e", "f"]])

proj = honet.clique_projection(hg)
for node in hg.nodes:
    print(f"{node}: 投影邻居={sorted(proj[node])}  逐点查询={sorted(hg.neighbors(node))}")

print()
print("每个结点都有条目（含孤立结点）:", set(proj) == set(hg.nodes))
print("投影是否对称:", all(n in proj[m] for m in proj for n in proj[m]))

a: 投影邻居=['b', 'c']  逐点查询=['b', 'c']
b: 投影邻居=['a', 'c']  逐点查询=['a', 'c']
c: 投影邻居=['a', 'b', 'd', 'e']  逐点查询=['a', 'b', 'd', 'e']
d: 投影邻居=['c', 'e']  逐点查询=['c', 'e']
e: 投影邻居=['c', 'd', 'f']  逐点查询=['c', 'd', 'f']
f: 投影邻居=['e']  逐点查询=['e']

每个结点都有条目（含孤立结点）: True
投影是否对称: True


## 高阶边数 → 投影边数（团膨胀）

一条 `k` 元超边展开后产生 `C(k, 2) = k(k-1)/2` 条成对边。
统计一下：原图有 3 条超边，投影后有多少条无向边。

In [2]:
pairs = {(x, y) for x, ys in proj.items() for y in ys if x < y}
expected = sum(e.size * (e.size - 1) // 2 for e in hg.edges)

print(f"超边数={hg.num_edges}，各边贡献的团内边数={[e.size * (e.size - 1) // 2 for e in hg.edges]}")
print(f"去重后投影边数={len(pairs)}（叠加值上限={expected}）")
print("投影边:", sorted(pairs))

超边数=3，各边贡献的团内边数=[3, 3, 1]
去重后投影边数=7（叠加值上限=7）
投影边: [('a', 'b'), ('a', 'c'), ('b', 'c'), ('c', 'd'), ('c', 'e'), ('d', 'e'), ('e', 'f')]


## 信息损失示例

下面两个超图**投影完全相同**，但一个记录了“三人同场”的高阶事件，另一个只有两两合作。
若后续分析只依赖投影，两者不可区分——这是必须先决定“保留高阶结构还是降维”的原因。

In [3]:
triple = honet.Hypergraph(edges=[["a", "b", "c"]])
pairwise = honet.Hypergraph(edges=[["a", "b"], ["b", "c"], ["a", "c"]])

p_triple = honet.clique_projection(triple)
p_pairwise = honet.clique_projection(pairwise)

print("三元边投影  :", {k: sorted(v) for k, v in p_triple.items()})
print("三条二元边投影:", {k: sorted(v) for k, v in p_pairwise.items()})
print("投影相同:", p_triple == p_pairwise)
print("但原图边数不同:", triple.num_edges, "vs", pairwise.num_edges)
print("原图是否相等:", triple == pairwise)

三元边投影  : {'a': ['b', 'c'], 'b': ['a', 'c'], 'c': ['a', 'b']}
三条二元边投影: {'a': ['b', 'c'], 'b': ['a', 'c'], 'c': ['a', 'b']}
投影相同: True
但原图边数不同: 1 vs 3
原图是否相等: False
